# Discretization (Binning) — `KBinsDiscretizer`

Where binarization split a column into just 2 buckets, **discretization** splits a continuous column into **several** ordered bins (here, 15) — a middle ground between raw continuous values and a single 0/1 cutoff. `Age` and `Fare` are each binned independently.

## 1. Setup and Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer

In [3]:
df = pd.read_csv('train.csv',usecols=['Age','Fare','Survived'])

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
X = df.iloc[:,1:]
y = df.iloc[:,0]

In [ ]:

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train.head(2)

## 2. Baseline — No Binning

In [ ]:

clf = DecisionTreeClassifier()

In [ ]:
clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)

In [ ]:

accuracy_score(y_test,y_pred)

In [ ]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

## 3. Binning `Age` and `Fare`

`n_bins=15` splits each column into 15 ordered groups. `encode='ordinal'` outputs the bin **index** (`0, 1, 2, ...`) rather than one-hot columns. `strategy='quantile'` sizes the bins so each contains **roughly equal numbers of rows** (rather than equal-width ranges) — sensible for a skewed column like `Fare`, where equal-width bins would leave almost everyone in the first bin and a handful of outliers spread across the rest.

In [ ]:
kbin_age = KBinsDiscretizer(n_bins=15,encode='ordinal',strategy='quantile')
kbin_fare = KBinsDiscretizer(n_bins=15,encode='ordinal',strategy='quantile')

In [ ]:
trf = ColumnTransformer([
    ('first',kbin_age,[0]),
    ('second',kbin_fare,[1])
])

In [ ]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

`bin_edges_` reveals the actual boundary values chosen for each bin.

In [ ]:
trf.named_transformers_['first'].bin_edges_

### ⚠️ Copy-Paste Bug: This Should Show `'second'` (Fare), Not `'first'` (Age) Again

The original notebook has this exact same cell twice in a row — almost certainly meant to show `Fare`'s bin edges the second time, not repeat `Age`'s.

In [ ]:
trf.named_transformers_['second'].bin_edges_


## 4. Comparing Original vs. Binned Values Side by Side

In [ ]:
output = pd.DataFrame({
    'age':X_train['Age'],
    'age_trf':X_train_trf[:,0],
    'fare':X_train['Fare'],
    'fare_trf':X_train_trf[:,1]
})

`pd.cut()` re-labels each row with the actual bin **range** it fell into (e.g. `(24.0, 28.0]`) — much more readable than a bare bin index when inspecting results.

In [ ]:
output['age_labels'] = pd.cut(x=X_train['Age'],
                                    bins=trf.named_transformers_['first'].bin_edges_[0].tolist())
output['fare_labels'] = pd.cut(x=X_train['Fare'],
                                    bins=trf.named_transformers_['second'].bin_edges_[0].tolist())



In [ ]:
output.sample(5)

### Visualizing the Bins

Plotting the binned column shows roughly equal-sized groups (confirming `strategy='quantile'` worked as intended), compared against the original continuous distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
X_train['Age'].hist(ax=axes[0], bins=30)
axes[0].set_title('Age — original (continuous)')
pd.Series(X_train_trf[:, 0]).value_counts().sort_index().plot(kind='bar', ax=axes[1])
axes[1].set_title('Age — after quantile binning (15 bins)')
plt.show()


## 5. Accuracy After Binning

In [ ]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

In [ ]:
accuracy_score(y_test,y_pred2)

### ⚠️ Bug: Cross-Validating on the *Untransformed* Data

`X_trf = trf.fit_transform(X)` computes the binned version, but the very next line passes plain `X` (the raw, unbinned data) into `cross_val_score` — so this comparison was accidentally testing the baseline again, not the binned result. Fixed below to actually use `X_trf`.

In [ ]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(), X_trf, y, cv=10, scoring='accuracy'))


## 6. Comparing Binning Strategies

`KBinsDiscretizer` supports three `strategy` options:
- **`'uniform'`** — every bin covers an equal-width range (simple, but skewed data leaves bins very unevenly populated)
- **`'quantile'`** — every bin gets roughly the same **number of rows** (used above)
- **`'kmeans'`** — bin edges are placed using 1D k-means clustering, grouping naturally similar values together

*(The original `discretize()` function was cut off mid-line — completed below.)*

In [ ]:
def discretize(bins, strategy):
    kbin_age = KBinsDiscretizer(n_bins=bins, encode='ordinal', strategy=strategy)
    kbin_fare = KBinsDiscretizer(n_bins=bins, encode='ordinal', strategy=strategy)

    trf = ColumnTransformer([
        ('first', kbin_age, [0]),
        ('second', kbin_fare, [1])
    ])

    X_trf = trf.fit_transform(X)
    print('Accuracy:', np.mean(cross_val_score(DecisionTreeClassifier(), X_trf, y, cv=10, scoring='accuracy')))


In [ ]:
discretize(5,'kmeans')

Comparing a few more combinations directly:

In [ ]:
discretize(10, 'quantile')
discretize(10, 'uniform')
discretize(10, 'kmeans')


## Summary

| Task | Method |
|---|---|
| Bin a continuous column into ordered groups | `KBinsDiscretizer(n_bins=k, encode='ordinal', strategy=...)` |
| Equal-count bins (good for skewed data) | `strategy='quantile'` |
| Equal-width bins | `strategy='uniform'` |
| Cluster-based bins | `strategy='kmeans'` |
| See the chosen boundaries | `column_transformer.named_transformers_['name'].bin_edges_` |
| Label rows with their bin range | `pd.cut(x=col, bins=edges)` |

### Watch out for
Always double-check that the *transformed* array (`X_trf`), not the original `X`, is what actually gets passed into `cross_val_score` / `.fit()` — an easy copy-paste slip to make after computing several variants.